# PV/ASOS 1단계 데이터 정제 — E0/E1 공통 표본 생성

**입력:** 이전 프로파일링의 `profile_output_B.zip` (내부 `05_plant_A_asos_aligned_original_timestamps.csv`, `05_plant_B_asos_aligned_original_timestamps.csv`). 별도의 원본 CSV가 아니라 **이미 기상 시각 정합된 결과**를 사용합니다.

**출력:** `clean_output/` 하위 행별 품질 플래그, 격리 행, E0/E1 공통 정제 데이터셋, 시간순 학습/검증/테스트 분할, 품질·특징 검증 보고서.

- 원본 시각은 유지합니다. 5분 고정 재표본화는 하지 않습니다.
- 이상 징후와 데이터 오류를 구분합니다. 고장코드 및 출력 0은 자동 제외하지 않습니다.
- 아직 **Isolation Forest 학습은 수행하지 않습니다.** 실제 고장 레이블이 없으므로 이상 탐지 정확도를 의미하지 않습니다.
- 입력 파생 컬럼 `delta_accu`, `rate_output`은 원본 값을 보존하고 계산 검증용으로 사용합니다. 모델에 사용할 시간 보정 특징은 별도로 생성합니다.
- 기상 특징이 누락된 데이터는 E0 전용으로 보존할 수 있지만, E0/E1 비교는 **동일한 공통 표본**으로 수행합니다.

> 주의: 임계값은 초기 품질검사 정책이며 제조사 사양과 실제 운전 조건에 따라 조정해야 합니다. 출력 비율이 1을 조금 초과했다고 곧바로 고장으로 판단하지 않습니다.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import io
import numpy as np
import pandas as pd
from IPython.display import display

ZIP_PATH = Path('profile_output_B.zip')
OUT_DIR = Path('clean_output')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 시간 간격은 원본에 따라 약 5분으로 변동. 차분/이동 특징의 유효 연속성 범위
MIN_INTERVAL_MIN = 2.0
MAX_INTERVAL_MIN = 8.0
# 분모 0, 비정상 전력값을 포함하지 않도록 '학습 가능' 범위만 제한
MAX_POWER_RATIO = 1.20
MAX_ENERGY_RATIO = 1.25
# 수집된 누적값의 실질 단위가 Wh임을 검증한 뒤 필요 시 변경
ACCUMULATED_ENERGY_TO_WH = 1.0
# 원본 delta_accu와 계산값 허용 오차
ENERGY_DELTA_TOL_WH = 1.0
# rolling 조건: 현재 및 직전 관측이 연속이고 정상적인 60분 창, 최소 4관측
ROLLING_WINDOW = '60min'
ROLLING_MIN_OBS = 4
# 발전 중 기준: 소출력 데이터는 야간/대기/저일사 상태 혼재로 별도 보관
MIN_ACTIVE_POWER_RATIO = 0.005
# E1 기상 특징: 사용 가능한 실제 컬럼을 자동 선택. 일사량/강수량은 포함하지 않음
WEATHER_FEATURES = ['asos__기온(°C)', 'asos__습도(%)', 'asos__현지기압(hPa)']
# 품질 이상 행을 직접 확인할 수 있도록 전부 별도 파일로 출력
SAVE_COMPLETE_AUDIT = True

assert ZIP_PATH.exists(), f'ZIP 파일이 없습니다: {ZIP_PATH.resolve()}'
print('입력 ZIP:', ZIP_PATH.resolve())

## 1. 보고서 ZIP에서 발전소별 정합 데이터 로드

CSV에 존재하는 원본 컬럼(발전출력, 누적 발전량, 파생 특징)과 ASOS 컬럼을 유지합니다. 헤더가 변경된 경우 오류를 알려 임의 컬럼에 매핑하지 않습니다.

In [ ]:
REQUIRED = ['reg_date', 'output', 'accu_energy', 'rate_output', 'delta_accu']
MEMBERS = {
    'A': '05_plant_A_asos_aligned_original_timestamps.csv',
    'B': '05_plant_B_asos_aligned_original_timestamps.csv',
}

def load_from_report_zip(zip_path, member, plant_id):
    with ZipFile(zip_path) as z:
        if member not in z.namelist():
            raise FileNotFoundError(f'ZIP 내부 파일 누락: {member}')
        with z.open(member) as f:
            df = pd.read_csv(f, low_memory=False)
    missing = sorted(set(REQUIRED) - set(df.columns))
    if missing:
        raise ValueError(f'{plant_id}: 필수 컬럼 없음: {missing}')
    timestamp_col = '_timestamp' if '_timestamp' in df.columns else 'reg_date'
    df['timestamp'] = pd.to_datetime(df[timestamp_col], errors='coerce')
    df['source_plant'] = plant_id
    df['source_row'] = np.arange(len(df), dtype=np.int64)
    for c in ['output','accu_energy','rate_output','delta_accu']:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    return df

plants = {p: load_from_report_zip(ZIP_PATH, m, p) for p,m in MEMBERS.items()}
for p,df in plants.items():
    print(p, '행:',len(df), '기간:',df.timestamp.min(), '~',df.timestamp.max())
print('기상 원본 컬럼:', [c for c in plants['A'].columns if c.startswith('asos__') and '__' not in c[6:]])

## 2. 타임스탬프·물리범위·누적값 차분 검증

- 중복은 삭제하기 전 **원본 순서대로 모두 플래그** 처리합니다. 중복 데이터 중 임의 하나를 정상으로 선택하지 않습니다.
- 차분은 인버터 식별키 (`sid`, `deviceID`, `multiID`)별로 계산하며, 직전 레코드가 같은 시각이면 무효로 판정합니다.
- 비정상 정격출력 대비 출력 비율 / 음수 발전량 등은 **데이터 품질 검토 대상**으로 격리합니다.
- `faultcode`는 인버터마다 정상 코드가 다를 수 있어 자동 제외 조건이 아닙니다.
- 간격이 유효하지 않으면 구간 발전량 비율과 변화량을 모델 특징으로 사용하지 않습니다.

In [ ]:
def make_audit(df):
    d = df.copy().sort_values(['timestamp','source_row'], kind='stable').reset_index(drop=True)
    key_cols = [c for c in ['sid','deviceID','multiID'] if c in d.columns]
    group_keys = ['source_plant'] + key_cols
    d['flag_invalid_timestamp'] = d.timestamp.isna()
    d['flag_duplicate_timestamp'] = d.duplicated(group_keys+['timestamp'], keep=False)
    d['interval_min'] = d.groupby(group_keys,dropna=False)['timestamp'].diff().dt.total_seconds().div(60)
    d['flag_invalid_interval'] = ~(d.interval_min.between(MIN_INTERVAL_MIN, MAX_INTERVAL_MIN))
    d['previous_accu'] = d.groupby(group_keys, dropna=False)['accu_energy'].shift(1)
    d['recomputed_delta_wh'] = (d.accu_energy - d.previous_accu) * ACCUMULATED_ENERGY_TO_WH
    d['flag_missing_core'] = d[['output','accu_energy','rate_output','delta_accu']].isna().any(axis=1)
    d['flag_negative_power'] = d.output < 0
    d['flag_negative_accu'] = d.accu_energy < 0
    d['flag_invalid_power_ratio'] = ~d.rate_output.between(0, MAX_POWER_RATIO)
    d['flag_negative_energy_delta'] = d.recomputed_delta_wh < 0
    d['flag_delta_mismatch'] = (d.delta_accu * ACCUMULATED_ENERGY_TO_WH - d.recomputed_delta_wh).abs() > ENERGY_DELTA_TOL_WH
    d['flag_delta_mismatch'] = d['flag_delta_mismatch'].fillna(False)
    # 저장된 파생 특징과의 계산 일관성: rated power column이 없으므로 정격출력 50/110kW를 추정하여 검증하는 것은 하지 않음
    d['energy_ratio'] = d.recomputed_delta_wh / (d.interval_min / 60) / np.where(d.rate_output > 0, d.output / d.rate_output, np.nan)
    # 무출력 구간의 energy_ratio는 위 방식으로 계산할 수 없으므로 이후에는 rated_power를 명시적으로 채택해야 함
    # 출력>0인 경우의 추정 rated_power 분포를 확인하고 설비별 정상값을 구해 적용
    d['rated_w_candidate'] = np.where(d.rate_output > 0, d.output / d.rate_output, np.nan)
    return d, group_keys

audits = {}
for p, df in plants.items():
    audits[p], _ = make_audit(df)
    print(p, '중복',int(audits[p].flag_duplicate_timestamp.sum()),
          '간격 부적합',int(audits[p].flag_invalid_interval.sum()),
          '비율 부적합',int(audits[p].flag_invalid_power_ratio.sum()),
          '음수 차분',int(audits[p].flag_negative_energy_delta.sum()))

## 3. 정격출력 추정 및 파생 특징 확인

보고서 데이터에는 정격출력 제원 컬럼이 포함되지 않았으므로, 저장된 `output / rate_output` 분포에서 정격출력 후보를 **검증용으로 추정**합니다. 추정치가 하나로 안정적인 경우에만 사용합니다. 원본 제원과 다르면 `RATED_POWER_OVERRIDE_W`로 직접 지정하십시오.

In [ ]:
RATED_POWER_OVERRIDE_W = {'A': None, 'B': None}  # 정확한 제조사 AC 정격출력[W]로 교체 가능
RATED_CV_LIMIT = 0.001
rated_rows = []
for p,d in audits.items():
    candidates = d.loc[(d.output>0)&d.rate_output.between(.01,1.05)&(~d.flag_duplicate_timestamp),'rated_w_candidate'].replace([np.inf,-np.inf],np.nan).dropna()
    median = float(candidates.median()) if len(candidates) else np.nan
    relative_mad = float((candidates-median).abs().median()/median) if median>0 else np.inf
    rated = RATED_POWER_OVERRIDE_W[p] if RATED_POWER_OVERRIDE_W[p] is not None else median
    if not np.isfinite(rated) or rated <= 0 or relative_mad > RATED_CV_LIMIT:
        raise ValueError(f'{p}: 정격출력 추정 불안정: median={median}, relative_MAD={relative_mad}. RATED_POWER_OVERRIDE_W 설정 필요')
    d['rated_power_w_used'] = rated
    d['power_ratio_recomputed'] = d.output / rated
    d['flag_power_ratio_mismatch'] = (d.power_ratio_recomputed-d.rate_output).abs()>0.00011
    d['energy_ratio'] = d.recomputed_delta_wh / (d.interval_min / 60) / rated
    d['flag_energy_ratio_outlier'] = ~d.energy_ratio.between(0,MAX_ENERGY_RATIO)
    d['flag_energy_ratio_outlier'] = d.flag_energy_ratio_outlier & ~d.flag_invalid_interval
    rated_rows.append(dict(plant=p, inferred_rated_w=median, used_rated_w=rated, relative_mad=relative_mad, sample_count=len(candidates)))
rated_summary=pd.DataFrame(rated_rows)
display(rated_summary)
rated_summary.to_csv(OUT_DIR/'01_rated_power_validation.csv',index=False,encoding='utf-8-sig')

## 4. 정상 학습 후보 선별 및 인버터별 시계열 특징

**원칙:** 물리적/수집 오류는 별도로 격리, 정상 운전 패턴 학습 대상은 `active`로 분류. 야간/정지는 원본 감사 파일에 유지합니다. 오류값에 의해 이동 특징이 전파되지 않도록 연속성이 깨지면 구간을 다시 시작합니다.

`faultcode`가 비정상이라는 가정은 하지 않습니다. 알려진 고장 발생 여부는 향후 별도 검증합니다.

In [ ]:
FLAGS_HARD = [
    'flag_invalid_timestamp','flag_duplicate_timestamp','flag_invalid_interval',
    'flag_missing_core','flag_negative_power','flag_negative_accu',
    'flag_invalid_power_ratio','flag_negative_energy_delta','flag_delta_mismatch',
    'flag_power_ratio_mismatch','flag_energy_ratio_outlier'
]

def clean_and_features(d):
    d=d.copy()
    d['flag_hard_quality'] = d[FLAGS_HARD].any(axis=1)
    d['flag_inactive'] = d.rate_output < MIN_ACTIVE_POWER_RATIO
    d['is_train_candidate'] = ~d.flag_hard_quality & ~d.flag_inactive
    # 최초 행/결측 뒤의 행이 시계열 특징을 잘못 이어받지 않도록 구간 분리
    keys = [c for c in ['source_plant','sid','deviceID','multiID'] if c in d.columns]
    d['segment_start'] = d['flag_hard_quality'] | d['flag_hard_quality'].groupby([d[c] for c in keys],sort=False).shift(1,fill_value=True)
    d['segment_id'] = d.groupby(keys,dropna=False)['segment_start'].cumsum()
    d['power_ratio_diff'] = d.groupby(keys+['segment_id'],dropna=False).rate_output.diff()
    d['ratio_ma_60min'] = np.nan
    # O(n log n) / per-segment vectorized algorithm instead of Python loops over thousands of segments
    valid = d.loc[~d.flag_hard_quality & d.timestamp.notna(), keys+['segment_id','timestamp','rate_output']].copy()
    if len(valid):
        valid['_ns']=valid.timestamp.astype('int64')
        valid['_groupid']=valid.groupby(keys+['segment_id'],sort=False,dropna=False).ngroup()
        # sort by group + timestamp, then window bounds using searchsorted per group
        valid=valid.sort_values(['_groupid','_ns'])
        vals=valid.rate_output.to_numpy(dtype=float)
        ts=valid._ns.to_numpy(dtype=np.int64)
        gids=valid._groupid.to_numpy(dtype=np.int64)
        starts=np.flatnonzero(np.r_[True,gids[1:]!=gids[:-1]])
        ends=np.r_[starts[1:],len(valid)]
        means=np.full(len(valid),np.nan)
        horizon=pd.Timedelta(ROLLING_WINDOW).value
        for a,b in zip(starts,ends):
            if b-a<ROLLING_MIN_OBS: continue
            ts_g=ts[a:b]; v_g=vals[a:b]
            left=np.searchsorted(ts_g,ts_g-horizon,side='right')
            count=np.arange(b-a)-left+1
            cum=np.r_[0.,np.cumsum(v_g)]
            out=(cum[np.arange(1,b-a+1)]-cum[left])/count
            out[count<ROLLING_MIN_OBS]=np.nan
            means[a:b]=out
        d.loc[valid.index,'ratio_ma_60min']=means
    d['flag_missing_temporal_features'] = d[['power_ratio_diff','ratio_ma_60min']].isna().any(axis=1)
    d['selected_e0'] = d.is_train_candidate & ~d.flag_missing_temporal_features
    weather_present = [c for c in WEATHER_FEATURES if c in d.columns]
    if len(weather_present)!=len(WEATHER_FEATURES):
        raise ValueError('기상 컬럼 누락: '+str(set(WEATHER_FEATURES)-set(weather_present)))
    for c in weather_present:
        d[c]=pd.to_numeric(d[c],errors='coerce')
    d['flag_weather_missing'] = d[weather_present].isna().any(axis=1)
    d['selected_common_e0_e1'] = d.selected_e0 & ~d.flag_weather_missing
    return d

cleaned={p:clean_and_features(d) for p,d in audits.items()}
for p,d in cleaned.items():
    print(p,'E0:',int(d.selected_e0.sum()),'E0/E1 공통:',int(d.selected_common_e0_e1.sum()),'격리:',int(d.flag_hard_quality.sum()))

## 5. 날짜 기준 train/validation/test 분할

분할은 **발전소별 원본 시계열** 순으로 70%/15%/15%를 사용합니다. 각 발전소 내부 동일 원본 관측 시각의 가상 복제가 학습/평가 양쪽에 들어가지 않도록, 가상 발전소 생성은 분할 후에 하십시오.

ASOS 자료에 오프라인 양방향 보간이 이미 포함되어 있어 미래값을 참조할 수 있으므로, 이 결과를 실제 온라인 예측 평가로 해석하면 안 됩니다.

In [ ]:
E0_FEATURES = ['rate_output','energy_ratio','power_ratio_diff','ratio_ma_60min']
E1_FEATURES = E0_FEATURES + WEATHER_FEATURES
SPLITS = (0.70,0.15,0.15)
assert abs(sum(SPLITS)-1)<1e-10

def assign_splits(d):
    d=d.copy()
    d['split']='excluded'
    good=d[d.selected_common_e0_e1].sort_values('timestamp')
    # 타임스탬프 기준 cutpoints: 두 발전소 관측 기간 차이를 유지하면서 순서 보장
    if len(good)<100: raise ValueError('학습 후보가 부족합니다. 데이터 품질 플래그 확인 필요.')
    unique_times=good.timestamp.drop_duplicates().sort_values().to_numpy()
    k1=int(len(unique_times)*SPLITS[0]);k2=int(len(unique_times)*(SPLITS[0]+SPLITS[1]))
    cut1=unique_times[k1]; cut2=unique_times[k2]
    d.loc[d.selected_common_e0_e1 & (d.timestamp < cut1),'split']='train'
    d.loc[d.selected_common_e0_e1 & d.timestamp.ge(cut1) & d.timestamp.lt(cut2),'split']='validation'
    d.loc[d.selected_common_e0_e1 & d.timestamp.ge(cut2),'split']='test'
    return d,cut1,cut2

splits={}
split_cuts=[]
for p,d in cleaned.items():
    splits[p],a,b=assign_splits(d)
    split_cuts.append({'plant':p,'train_to':str(a),'validation_to':str(b)})
    print(p,splits[p].split.value_counts().to_dict())
pd.DataFrame(split_cuts).to_csv(OUT_DIR/'02_split_boundaries.csv',index=False,encoding='utf-8-sig')

## 6. 결과 보고서 및 E0/E1 공통 데이터 저장

`03_quality_summary.csv`: 각 단계의 행 수 및 품질 플래그 합계. `04_quality_quarantine.csv`: 원본 행 식별자 및 격리 사유. `05_common_E0_E1_clean.csv`: 두 모델이 동일 표본을 사용하도록 정제한 공통 데이터. 학습·검증·테스트 세트는 별도 저장됩니다.

**주의:** `E0`과 `E1`은 학습 단계에서 이 CSV의 같은 행을 사용하고, 각 모델에 입력하는 특징 컬럼만 다르게 해야 비교가 공정합니다.

In [ ]:
quality=[]; quarantine=[]; model_parts=[]
for p,d in splits.items():
    quality.append({
        'plant':p,'all_rows':len(d),'hard_quality_rows':int(d.flag_hard_quality.sum()),
        'inactive_rows':int(d.flag_inactive.sum()),
        'e0_rows':int(d.selected_e0.sum()),'common_e0_e1_rows':int(d.selected_common_e0_e1.sum()),
        'weather_missing_from_e0':int((d.selected_e0 & d.flag_weather_missing).sum()),
        **{c:int(d[c].sum()) for c in FLAGS_HARD},
        **{f'split_{name}':int((d.split==name).sum()) for name in ['train','validation','test']}
    })
    q=d.loc[d.flag_hard_quality | d.flag_missing_temporal_features | d.flag_inactive | d.flag_weather_missing].copy()
    reason_cols=FLAGS_HARD+['flag_missing_temporal_features','flag_inactive','flag_weather_missing']
    q['quality_reasons']=q[reason_cols].apply(lambda row: ';'.join(row.index[row.to_numpy(dtype=bool)]),axis=1)
    quarantine.append(q[['source_plant','source_row','timestamp','quality_reasons','output','rate_output','accu_energy','delta_accu','interval_min','recomputed_delta_wh','energy_ratio']])
    m=d.loc[d.selected_common_e0_e1, ['source_plant','source_row','timestamp','split','output','accu_energy','delta_accu','rated_power_w_used','interval_min']+E1_FEATURES].copy()
    model_parts.append(m)
    if SAVE_COMPLETE_AUDIT:
        d.to_csv(OUT_DIR/f'03_plant_{p}_full_audit.csv',index=False,encoding='utf-8-sig')
summary=pd.DataFrame(quality)
summary.to_csv(OUT_DIR/'03_quality_summary.csv',index=False,encoding='utf-8-sig')
qall=pd.concat(quarantine,ignore_index=True)
qall.to_csv(OUT_DIR/'04_quality_quarantine.csv',index=False,encoding='utf-8-sig')
model=pd.concat(model_parts,ignore_index=True).sort_values(['timestamp','source_plant','source_row'])
model.to_csv(OUT_DIR/'05_common_E0_E1_clean.csv',index=False,encoding='utf-8-sig')
for name in ['train','validation','test']:
    model.loc[model.split.eq(name)].to_csv(OUT_DIR/f'06_common_{name}.csv',index=False,encoding='utf-8-sig')
display(summary.T)
print('공통 정제 데이터 행:',len(model))
print('결과 위치:',OUT_DIR.resolve())

## 7. 최종 자동 검증

가장 중요한 보장 조건: 모든 행의 인버터 출력 시각을 보존하고, E0/E1 후보와 분할 파일의 일치, 시간순 분할, 특징 비결측 여부를 확인합니다.

In [ ]:
assert len(model)==sum(len(part) for part in model_parts)
assert model[E1_FEATURES].notna().all().all(), 'E1 입력에 결측 존재'
assert np.isfinite(model[E1_FEATURES].to_numpy(dtype=float)).all(), 'E1 특징에 무한값 존재'
assert model[['source_plant','source_row']].duplicated().sum()==0, '원본 행 중복'
assert set(model.split.unique())=={'train','validation','test'}, '분할 오류'
for p,g in model.groupby('source_plant'):
    ts=g.groupby('split').timestamp.agg(['min','max'])
    assert ts.loc['train','max'] < ts.loc['validation','min'] < ts.loc['test','min'], f'{p} 시간 분할 순서 오류'
    assert ts.loc['validation','max'] < ts.loc['test','min'],f'{p} 검증/테스트 경계 오류'
print('PASS: E0/E1 공통 표본, 결측/무한값, 원본 식별자, 시간 분할 검증 완료')
print('※ 실측 이상 레이블 검증과 실제 Python 3.10.20 환경 검증은 별도로 필요합니다.')